In [ ]:
# All the following cells should print Ok. If they don't we have a problem.
import errno
import os
import shutil

ROOT = "rmdir_test"

shutil.rmtree(ROOT, ignore_errors=True)
os.mkdir(ROOT)

print("Ok")

In [ ]:
# rmdir of a directory containing a file fails and leaves it untouched
dirname = os.path.join(ROOT, "with_file")
filename = os.path.join(dirname, "a.txt")
content = "Hello there"

os.mkdir(dirname)
with open(filename, "w") as fobj:
    fobj.write(content)

try:
    os.rmdir(dirname)
    raise AssertionError("rmdir of non-empty directory succeeded")
except OSError as e:
    assert e.errno == errno.ENOTEMPTY, f"Unexpected errno {e.errno}"

assert os.path.isdir(dirname)
with open(filename, "r") as fobj:
    assert fobj.read() == content

print("Ok")

In [ ]:
# rmdir of a directory containing only an empty subdirectory fails
dirname = os.path.join(ROOT, "with_subdir")
subdirname = os.path.join(dirname, "sub")

os.makedirs(subdirname)

try:
    os.rmdir(dirname)
    raise AssertionError("rmdir of non-empty directory succeeded")
except OSError as e:
    assert e.errno == errno.ENOTEMPTY, f"Unexpected errno {e.errno}"

assert os.path.isdir(subdirname)

print("Ok")

In [ ]:
# rmdir of an empty directory succeeds
dirname = os.path.join(ROOT, "empty")

os.mkdir(dirname)
os.rmdir(dirname)

assert not os.path.exists(dirname)

print("Ok")

In [ ]:
# Removing a file then trying to rmdir its parent keeps the siblings, as libgit2 checkout does
dirname = os.path.join(ROOT, "src")

os.mkdir(dirname)
for name in ("css.d.ts", "defs.ts", "index.ts"):
    with open(os.path.join(dirname, name), "w") as fobj:
        fobj.write(name)

os.unlink(os.path.join(dirname, "css.d.ts"))
try:
    os.rmdir(dirname)
except OSError:
    pass

assert os.path.isdir(dirname)
assert sorted(os.listdir(dirname)) == ["defs.ts", "index.ts"], os.listdir(dirname)

print("Ok")

In [ ]:
# Recursive removal of a populated tree still works
dirname = os.path.join(ROOT, "tree")

os.makedirs(os.path.join(dirname, "a", "b"))
for filename in (os.path.join(dirname, "a", "b", "f.txt"), os.path.join(dirname, "g.txt")):
    with open(filename, "w") as fobj:
        fobj.write("x")

shutil.rmtree(dirname)

assert not os.path.exists(dirname)

print("Ok")

In [ ]:
# Clean up
shutil.rmtree(ROOT)

assert not os.path.exists(ROOT)

print("Ok")